# Hands-on AI in Healthcare - Chapter 5: Neural Networks

## 5.1 The Note Router: Setting Up

### 5.1.1 Installing and Importing Libraries

In [ ]:
# Install the libraries we'll use in this chapter.
# PyTorch (torch) is new: it is the library we'll use to build neural networks.
%pip install pandas matplotlib scikit-learn numpy torch

In [ ]:
import glob
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import torch
import torch.nn as nn

print(f"pandas      : {pd.__version__}")
print(f"matplotlib  : {plt.matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"torch       : {torch.__version__}")
print("\nAll libraries loaded successfully!")

In [ ]:
# Set every random seed so results are reproducible from run to run.
SEED = 33
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### 5.1.2 About the Data

Download `mtsamples.csv` from Kaggle ([tboyle10/medicaltranscriptions](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions)) and place it in the `working/` directory next to this notebook:

```
chapter_5.ipynb
working/
    mtsamples.csv
```

In [ ]:
# Look for the MTSamples CSV in the working/ folder.
csv_files = sorted(glob.glob("working/mtsamples*.csv"))
if not csv_files:
    raise FileNotFoundError(
        "No mtsamples*.csv file found in the working/ folder.\n"
        "Please download it from "
        "https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions"
    )

csv_path = csv_files[-1]
print(f"Loading: {csv_path}")

notes = pd.read_csv(csv_path, index_col=0)

# The specialty labels have stray spaces around them; clean them up.
notes["medical_specialty"] = notes["medical_specialty"].str.strip()

# A few rows have no note text at all; drop them.
notes = notes.dropna(subset=["transcription"])

print(f"Dataset: {notes.shape[0]:,} notes x {notes.shape[1]} columns")

In [ ]:
# Look at one note. It should feel familiar after Chapter 4.
sample = notes.iloc[0]
print(f"Specialty:   {sample['medical_specialty']}")
print(f"Sample name: {sample['sample_name'].strip()}")
print()
print(sample["transcription"][:800] + " ...")

### [BONUS] Exploring the Dataset

This section isn't printed in the book. Before we filter the data down, it's worth seeing what the raw dataset actually looks like — how many notes each category holds, and how long a typical note runs. The two cells below draw those pictures.

In [ ]:
# How many notes does each of the 40 raw categories have?
counts = notes["medical_specialty"].value_counts()

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(counts.index[::-1], counts.values[::-1], color="steelblue")
ax.set_xlabel("Number of notes", fontsize=12)
ax.set_title("MTSamples: Notes per Category", fontsize=14)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

Look closely at the category names. Many of them are not medical specialties at all —
"Surgery", "SOAP / Chart / Progress Notes", "Office Notes", and "Discharge Summary" are
*document types* that could come from almost any specialty. And the counts are wildly
imbalanced, from over 1,000 notes down to a handful. Real-world data is messy; part of
the job is deciding what to keep.

In [ ]:
# How long is a typical note?
lengths = notes["transcription"].str.split().str.len()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(lengths, bins=60, color="lightblue", edgecolor="steelblue")
ax.set_xlabel("Words per note", fontsize=12)
ax.set_ylabel("Number of notes", fontsize=12)
ax.set_title("Distribution of Note Lengths", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Median note length: {lengths.median():.0f} words")

### 5.1.3 Filtering to Five Specialties

In [ ]:
SPECIALTIES = [
    "Cardiovascular / Pulmonary",
    "Orthopedic",
    "Gastroenterology",
    "Neurology",
    "Obstetrics / Gynecology",
]

notes5 = notes[notes["medical_specialty"].isin(SPECIALTIES)].copy()

print(notes5["medical_specialty"].value_counts())
print(f"\nTotal: {len(notes5):,} notes")

### 5.1.4 Splitting into Training and Test Sets

In [ ]:
train_notes, test_notes = train_test_split(
    notes5,
    test_size=0.3,
    stratify=notes5["medical_specialty"],  # keep the specialty mix the same in both sets
    random_state=SEED,
)

print(f"Training notes: {len(train_notes)}")
print(f"Test notes:     {len(test_notes)}")

---

## 5.2 From Words to Vectors

### 5.2.1 A Tiny Bag of Words

In [ ]:
# Three tiny "notes" to see how text becomes numbers.
tiny_notes = [
    "patient reports chest pain",
    "patient denies chest pain",
    "knee pain after a fall",
]

tiny_vectorizer = CountVectorizer()
tiny_vectors = tiny_vectorizer.fit_transform(tiny_notes)

pd.DataFrame(
    tiny_vectors.toarray(),
    columns=tiny_vectorizer.get_feature_names_out(),
    index=[f"note {i + 1}" for i in range(len(tiny_notes))],
)

### 5.2.2 TF-IDF: Weighting the Words That Matter

In [ ]:
# Turn every note into a 3,000-number vector.
vectorizer = TfidfVectorizer(
    max_features=3000,     # keep the 3,000 most frequent words
    stop_words="english",  # drop filler words like "the" and "of"
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",  # words only, no bare numbers
)

# Learn the vocabulary from the training notes only, then apply it to both sets.
X_train = vectorizer.fit_transform(train_notes["transcription"])
X_test = vectorizer.transform(test_notes["transcription"])

y_train = train_notes["medical_specialty"].values
y_test = test_notes["medical_specialty"].values

print(f"Training matrix: {X_train.shape[0]} notes x {X_train.shape[1]} features")
print(f"Test matrix:     {X_test.shape[0]} notes x {X_test.shape[1]} features")

### 5.2.3 Specialty Fingerprints

In [ ]:
# Distinctive words: each specialty's average TF-IDF minus the overall average.
vocab = vectorizer.get_feature_names_out()
overall_mean = np.asarray(X_train.mean(axis=0)).ravel()

for specialty in SPECIALTIES:
    mask = y_train == specialty
    class_mean = np.asarray(X_train[mask].mean(axis=0)).ravel()
    top = (class_mean - overall_mean).argsort()[::-1][:8]
    words = ", ".join(vocab[j] for j in top)
    print(f"{specialty:28s} -> {words}")

---

## 5.3 A Neuron Is a Regression with a Squish

### 5.3.1 The Sigmoid Activation

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


z = np.linspace(-8, 8, 200)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(z, sigmoid(z), color="red", linewidth=2)
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("z (the weighted sum)", fontsize=12)
ax.set_ylabel("sigmoid(z)", fontsize=12)
ax.set_title("The Sigmoid: Any Number In, a Probability Out", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3.2 A Two-Specialty Warm-Up

In [ ]:
BINARY = ["Cardiovascular / Pulmonary", "Orthopedic"]

train_mask = np.isin(y_train, BINARY)
test_mask = np.isin(y_test, BINARY)

# Dense arrays are easier to work with for hand-rolled math.
Xb_train = X_train[train_mask].toarray()
yb_train = (y_train[train_mask] == BINARY[0]).astype(float)
Xb_test = X_test[test_mask].toarray()
yb_test = (y_test[test_mask] == BINARY[0]).astype(float)

print(f"Training notes: {len(yb_train)} ({int(yb_train.sum())} cardiovascular/pulmonary)")
print(f"Test notes:     {len(yb_test)}")

### 5.3.3 Training One Neuron with Gradient Descent

In [ ]:
# One neuron: 3,000 weights (one per word) and one bias.
np.random.seed(SEED)
w = np.random.randn(Xb_train.shape[1]) * 0.01
b = 0.0

learning_rate = 1.0
n_iterations = 300
history = {"iteration": [], "loss": []}

for i in range(n_iterations):
    z = Xb_train @ w + b   # weighted sum -- just like multiple regression
    p = sigmoid(z)         # activation squishes it into a probability

    # Log loss: punishes confident wrong answers.
    loss = -np.mean(
        yb_train * np.log(p + 1e-9) + (1 - yb_train) * np.log(1 - p + 1e-9)
    )

    error = p - yb_train
    dw = Xb_train.T @ error / len(yb_train)  # gradient for each weight
    db = error.mean()                        # gradient for the bias

    w -= learning_rate * dw  # step against the gradient, scaled by the learning rate
    b -= learning_rate * db

    history["iteration"].append(i)
    history["loss"].append(loss)

print(f"Final training loss: {history['loss'][-1]:.4f}")

In [ ]:
# How well does one neuron do on notes it has never seen?
p_test = sigmoid(Xb_test @ w + b)
predictions = (p_test > 0.5).astype(float)
neuron_accuracy = (predictions == yb_test).mean()
print(f"One-neuron accuracy on unseen notes: {neuron_accuracy:.1%}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history["iteration"], history["loss"], color="steelblue", linewidth=2)
ax.set_xlabel("Iteration", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("One Neuron Learning to Tell Cardiology from Orthopedics", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5.4 Stacking Neurons: Layers and Networks

### 5.4.1 Counting Parameters

In [ ]:
n_features = 3000
n_hidden = 64
n_classes = len(SPECIALTIES)

hidden_params = n_features * n_hidden + n_hidden  # weights + biases
output_params = n_hidden * n_classes + n_classes

print(f"Hidden layer: {n_features:,} inputs x {n_hidden} neurons + {n_hidden} biases = {hidden_params:,}")
print(f"Output layer: {n_hidden} inputs x {n_classes} neurons + {n_classes} biases = {output_params:,}")
print(f"Total parameters: {hidden_params + output_params:,}")
print()
print("Chapter 3's multiple regression had 5.")

---

## 5.5 Training the Network: Backpropagation

In [ ]:
# Fix an order for the classes: class 0, class 1, ...
CLASS_NAMES = sorted(SPECIALTIES)
class_to_index = {name: i for i, name in enumerate(CLASS_NAMES)}

X_train_t = torch.tensor(X_train.toarray(), dtype=torch.float32)
y_train_t = torch.tensor([class_to_index[s] for s in y_train], dtype=torch.long)
X_test_t = torch.tensor(X_test.toarray(), dtype=torch.float32)
y_test_t = torch.tensor([class_to_index[s] for s in y_test], dtype=torch.long)

print(f"X_train tensor: {tuple(X_train_t.shape)}")
print(f"y_train tensor: {tuple(y_train_t.shape)}")

In [ ]:
# The whole network. Compare it, line by line, to the layer diagram in the chapter.
torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Linear(3000, 64),  # input layer -> hidden layer: 64 pattern detectors
    nn.ReLU(),            # activation: negative -> 0, positive -> unchanged
    nn.Linear(64, 5),     # hidden layer -> output layer: one score per specialty
)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

In [ ]:
loss_fn = nn.CrossEntropyLoss()  # like MSE, but for classification: punishes confident wrong answers
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # gradient descent with adaptive step sizes

n_epochs = 20    # one epoch = one full pass through the training notes
batch_size = 32  # update the weights after every 32 notes

train_losses = []
for epoch in range(n_epochs):
    model.train()
    permutation = torch.randperm(len(X_train_t))  # shuffle the notes each epoch
    epoch_losses = []
    for start in range(0, len(X_train_t), batch_size):
        batch = permutation[start:start + batch_size]

        logits = model(X_train_t[batch])          # forward pass: make predictions
        loss = loss_fn(logits, y_train_t[batch])  # measure how wrong they are

        optimizer.zero_grad()
        loss.backward()   # backpropagation: assign blame to every parameter
        optimizer.step()  # nudge all 192,389 of them

        epoch_losses.append(loss.item())
    train_losses.append(np.mean(epoch_losses))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:2d}: average training loss = {train_losses[-1]:.4f}")

In [ ]:
# Watch the network learn -- the same falling curve as Chapter 3.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, n_epochs + 1), train_losses, color="steelblue", linewidth=2, marker="o")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Average training loss", fontsize=12)
ax.set_title("The Network Learning to Read Clinical Notes", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5.6 Did It Work? Evaluating the Classifier

In [ ]:
model.eval()
with torch.no_grad():  # no gradients needed when we're only predicting
    test_logits = model(X_test_t)

predicted = test_logits.argmax(dim=1)
accuracy = (predicted == y_test_t).float().mean().item()
print(f"Accuracy on {len(y_test_t)} unseen notes: {accuracy:.1%}")

In [ ]:
cm = confusion_matrix(y_test_t.numpy(), predicted.numpy())
short_names = [name.split(" /")[0] for name in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(short_names)))
ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names, fontsize=10)

for i in range(len(short_names)):
    for j in range(len(short_names)):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=11,
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im, ax=ax, label="Number of notes", shrink=0.8)
ax.set_xlabel("Predicted specialty", fontsize=12)
ax.set_ylabel("Actual specialty", fontsize=12)
ax.set_title("Confusion Matrix: Where the Network Gets It Wrong", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Look at a few notes the network got wrong.
wrong = np.where((predicted != y_test_t).numpy())[0]
print(f"{len(wrong)} of {len(y_test_t)} test notes were misclassified.\n")

for idx in wrong[:3]:
    actual = CLASS_NAMES[y_test_t[idx]]
    guess = CLASS_NAMES[predicted[idx]]
    description = str(test_notes.iloc[idx]["description"]).strip()
    print(f"Actual: {actual}  |  Predicted: {guess}")
    print(f"   {description[:110]}")
    print()

### Route Your Own Note

In [ ]:
def classify_note(text):
    vector = vectorizer.transform([text]).toarray()
    with torch.no_grad():
        logits = model(torch.tensor(vector, dtype=torch.float32))
        probabilities = torch.softmax(logits, dim=1).ravel()  # softmax: scores -> probabilities
    print(f'"{text[:70]}..."' if len(text) > 70 else f'"{text}"')
    for i in probabilities.argsort(descending=True):
        print(f"   {CLASS_NAMES[i]:28s} {probabilities[i]:6.1%}")
    print()


classify_note(
    "Patient presents with crushing substernal chest pain radiating to the left "
    "arm. EKG shows ST elevation. Emergent cardiac catheterization was performed."
)

classify_note(
    "Right knee pain after a fall. MRI shows a torn medial meniscus. We discussed "
    "arthroscopic repair and physical therapy options."
)

### [BONUS] What the Network Pays Attention To

Compared to Chapter 3's regression, whose five weights we could simply read, this
network is a black box. But we can poke at it: feed it 3,000 fake "notes" that each
contain exactly one word, and see which specialty each word excites.

In [ ]:
# An identity matrix is 3,000 one-word "notes": note j contains only word j.
with torch.no_grad():
    word_logits = model(torch.eye(len(vocab)))

for class_index, name in enumerate(CLASS_NAMES):
    top_words = word_logits[:, class_index].argsort(descending=True)[:10]
    words = ", ".join(vocab[i] for i in top_words)
    print(f"{name:28s} -> {words}")

The network was never told what a stent or a meniscus is — it learned which words
matter for which specialty purely from labeled examples. This peek doesn't fully open
the black box (words interact inside the hidden layer), but it's reassuring that what
the network learned looks like medicine, not noise.

---

## 5.7 Hyperparameters: The Knobs on the Machine

In [ ]:
def train_network(X_tr, y_tr, hidden_size=64, n_classes=5, n_epochs=20,
                  learning_rate=0.001, batch_size=32, X_val=None, y_val=None):
    """Build and train a small network; optionally track validation loss per epoch."""
    torch.manual_seed(SEED)
    net = nn.Sequential(
        nn.Linear(X_tr.shape[1], hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, n_classes),
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)

    history = {"train_loss": [], "val_loss": []}
    for epoch in range(n_epochs):
        net.train()
        permutation = torch.randperm(len(X_tr))
        epoch_losses = []
        for start in range(0, len(X_tr), batch_size):
            batch = permutation[start:start + batch_size]
            loss = loss_fn(net(X_tr[batch]), y_tr[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        history["train_loss"].append(np.mean(epoch_losses))
        if X_val is not None:
            net.eval()
            with torch.no_grad():
                history["val_loss"].append(loss_fn(net(X_val), y_val).item())
    return net, history


def network_accuracy(net, X, y):
    net.eval()
    with torch.no_grad():
        return (net(X).argmax(dim=1) == y).float().mean().item()

### 5.7.1 How Many Hidden Neurons?

In [ ]:
hidden_sizes = [4, 16, 64, 256]
accuracies = []

for size in hidden_sizes:
    net, _ = train_network(X_train_t, y_train_t, hidden_size=size)
    acc = network_accuracy(net, X_test_t, y_test_t)
    accuracies.append(acc)
    print(f"Hidden size {size:3d}: test accuracy = {acc:.1%}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hidden_sizes, [a * 100 for a in accuracies], "o-",
        color="steelblue", linewidth=2, markersize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(hidden_sizes)
ax.set_xticklabels(hidden_sizes)
ax.set_xlabel("Hidden layer size (neurons)", fontsize=12)
ax.set_ylabel("Test accuracy (%)", fontsize=12)
ax.set_title("Bigger Isn't Always Better", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.7.2 Training Too Long: Overfitting

In [ ]:
_, long_history = train_network(
    X_train_t, y_train_t, n_epochs=150, X_val=X_test_t, y_val=y_test_t
)

fig, ax = plt.subplots(figsize=(9, 5))
epochs = range(1, len(long_history["train_loss"]) + 1)
ax.plot(epochs, long_history["train_loss"], color="steelblue", linewidth=2,
        linestyle="-", label="Training loss")
ax.plot(epochs, long_history["val_loss"], color="coral", linewidth=2,
        linestyle="--", label="Test loss")  # dashed so it reads without color
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Overfitting: The Network Starts Memorizing", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### [BONUS] The 40-Category Monster

What if we hadn't filtered to five specialties? Same architecture, same code — all the
raw categories (minus the tiniest, so the split works).

In [ ]:
# Keep every category with at least 10 notes so a stratified split is possible.
counts_all = notes["medical_specialty"].value_counts()
categories_all = sorted(counts_all[counts_all >= 10].index)
notes_all = notes[notes["medical_specialty"].isin(categories_all)]
print(f"Categories: {len(categories_all)}, notes: {len(notes_all):,}")

train_a, test_a = train_test_split(
    notes_all, test_size=0.3,
    stratify=notes_all["medical_specialty"], random_state=SEED,
)

vectorizer_all = TfidfVectorizer(
    max_features=3000, stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)
Xa_train = torch.tensor(
    vectorizer_all.fit_transform(train_a["transcription"]).toarray(),
    dtype=torch.float32,
)
Xa_test = torch.tensor(
    vectorizer_all.transform(test_a["transcription"]).toarray(),
    dtype=torch.float32,
)
index_all = {name: i for i, name in enumerate(categories_all)}
ya_train = torch.tensor(
    [index_all[s] for s in train_a["medical_specialty"]], dtype=torch.long
)
ya_test = torch.tensor(
    [index_all[s] for s in test_a["medical_specialty"]], dtype=torch.long
)

net_all, _ = train_network(Xa_train, ya_train, n_classes=len(categories_all))
acc_all = network_accuracy(net_all, Xa_test, ya_test)

print(f"\nAccuracy across {len(categories_all)} categories: {acc_all:.1%}")
print(f"Accuracy across our 5 specialties:  {accuracy:.1%}")

Accuracy craters — and it's not (mostly) the network's fault. "Surgery" overlaps with
every surgical specialty, "SOAP / Chart / Progress Notes" is a document type that could
be about anything, and dozens of categories have too few examples to learn from. The
lesson: **label quality and class design matter as much as the model.** In healthcare
AI projects, this is where a big share of the real work lives.

---

## 5.8 Three Ways to Learn

### 5.8.1 Unsupervised: Clustering Without Labels

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=SEED, n_init=10)
clusters = kmeans.fit_predict(X_train)

# Now reveal the labels: what actually landed in each cluster?
comparison = pd.crosstab(
    pd.Series(clusters, name="Cluster"),
    pd.Series([s.split(" /")[0] for s in y_train], name="Actual specialty"),
)
comparison

In [ ]:
# Squash the 3,000-dimensional vectors down to 2 dimensions so we can look at them.
pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(X_train.toarray())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distinct marker shapes so the groups read even in grayscale, not by color alone.
markers = ["o", "s", "^", "D", "v"]

for cluster_id in range(5):
    mask = clusters == cluster_id
    axes[0].scatter(coords[mask, 0], coords[mask, 1], s=18, alpha=0.7,
                    marker=markers[cluster_id], label=f"Cluster {cluster_id}")
axes[0].set_title("Grouped by K-Means cluster (no labels used)", fontsize=12)

for k, specialty in enumerate(CLASS_NAMES):
    mask = y_train == specialty
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=18, alpha=0.7,
                    marker=markers[k], label=specialty.split(" /")[0])
axes[1].set_title("Grouped by actual specialty", fontsize=12)

for ax in axes:
    ax.set_xlabel("PCA dimension 1")
    ax.set_ylabel("PCA dimension 2")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("The Same Notes, Grouped Without Labels vs. With Labels", fontsize=14)
plt.tight_layout()
plt.show()

### 5.8.2 Semi-Supervised: A Few Labels Go a Long Way

In [ ]:
# Pretend we could only afford to label 5% of the training notes.
labeled_idx, unlabeled_idx = train_test_split(
    np.arange(len(y_train_t)),
    train_size=0.05,
    stratify=y_train_t.numpy(),
    random_state=SEED,
)
print(f"Labeled notes:   {len(labeled_idx)}")
print(f"Unlabeled notes: {len(unlabeled_idx)} (labels hidden)")

# With so few notes, an epoch is only two batches -- so train for more epochs.
net_small, _ = train_network(X_train_t[labeled_idx], y_train_t[labeled_idx],
                             n_epochs=100)
acc_small = network_accuracy(net_small, X_test_t, y_test_t)
print(f"\nTest accuracy with 5% of the labels: {acc_small:.1%}")

In [ ]:
# Rounds of pseudo-labeling: label confidently, retrain, repeat.
net_current = net_small
for round_number in range(1, 4):
    # The current model labels the notes we couldn't afford to label...
    net_current.eval()
    with torch.no_grad():
        probabilities = torch.softmax(net_current(X_train_t[unlabeled_idx]), dim=1)
    confidence, pseudo_labels = probabilities.max(dim=1)
    confident_mask = confidence > 0.7  # ...and we keep only its confident calls.

    # Peek at the hidden labels to see how good the pseudo-labels are.
    agreement = (
        pseudo_labels[confident_mask]
        == y_train_t[unlabeled_idx][confident_mask]
    ).float().mean()

    # Retrain from scratch on real labels + pseudo-labels combined.
    X_combined = torch.cat(
        [X_train_t[labeled_idx], X_train_t[unlabeled_idx][confident_mask]]
    )
    y_combined = torch.cat(
        [y_train_t[labeled_idx], pseudo_labels[confident_mask]]
    )
    net_current, _ = train_network(X_combined, y_combined, n_epochs=100)

    acc_pseudo = network_accuracy(net_current, X_test_t, y_test_t)
    print(f"Round {round_number}: {confident_mask.sum().item():3d} pseudo-labels "
          f"({agreement:.0%} of them correct) -> test accuracy {acc_pseudo:.1%}")

In [ ]:
# Compare the three training recipes.
recipes = ["5% of labels", "5% + pseudo-labels", "100% of labels"]
scores = [acc_small, acc_pseudo, accuracy]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(recipes, [s * 100 for s in scores],
              color=["coral", "orange", "steelblue"], edgecolor="black")
for bar, hatch in zip(bars, ["//", "..", "xx"]):
    bar.set_hatch(hatch)  # distinct fill patterns so the bars differ without color
for bar, s in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f"{s:.1%}", ha="center", fontsize=12)
ax.set_ylabel("Test accuracy (%)", fontsize=12)
ax.set_ylim(0, 100)
ax.set_title("Semi-Supervised Learning: Winning Accuracy Back Without New Labels",
             fontsize=13)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---

## 5.9 Putting It Together

In [ ]:
# Route a stack of incoming notes.
np.random.seed(SEED)
stack = np.random.choice(len(test_notes), size=8, replace=False)

with torch.no_grad():
    probabilities = torch.softmax(model(X_test_t[stack]), dim=1)
confidence, routed = probabilities.max(dim=1)

routing = pd.DataFrame({
    "Incoming note": [str(test_notes.iloc[i]["description"]).strip()[:55] for i in stack],
    "Routed to": [CLASS_NAMES[c].split(" /")[0] for c in routed],
    "Confidence": [f"{c:.0%}" for c in confidence],
    "Actual": [test_notes.iloc[i]["medical_specialty"].split(" /")[0] for i in stack],
})
routing